# Artificial Neural Network

**The big picture:** the data is 10,000 customers of a bank. For each one we know things like their credit score, country, gender, age, account balance and salary, and whether they **left the bank** (*Exited* = 1) or stayed (0). The goal is to predict who is about to leave.

An **artificial neural network (ANN)** is a stack of small calculators called **neurons**, arranged in layers. Each neuron takes some numbers in, gives each one a weight (how much it matters), adds them up and passes a result on to the next layer. At first the weights are random. During training the network looks at thousands of customers, checks how wrong its guesses were, and nudges every weight a tiny bit to be less wrong. Think of tuning hundreds of little knobs until the answers come out right.

### Before you start
- **Read first:** **data_preprocessing_tools** (label encoding, one-hot encoding and feature scaling are all reused here) and **logistic_regression** (the sigmoid output and the 0.5 cut-off work the same way).
- **New words in this notebook:** *neuron*, *hidden layer*, *activation function (relu, sigmoid)*, *epoch*, *batch size*, *loss (binary cross-entropy)*. Look them up in the [glossary](../../../GLOSSARY.md) if you get stuck.

### Importing the libraries

- **numpy** does fast maths on lists of numbers.
- **pandas** reads and handles tables, like Excel in Python.
- **tensorflow** is Google's deep-learning library. Its **Keras** part (`tf.keras`) gives simple building blocks for neural networks.
- `tf.__version__` just shows which version is installed (2.2.0 here). Newer versions run this code too.

In [0]:
import numpy as np
import pandas as pd
import tensorflow as tf

In [2]:
tf.__version__

'2.2.0'

## Part 1 - Data Preprocessing

Before a neural network can learn, the data must be **all numbers** and **on similar scales**. This part gets the table into that shape.

### Importing the dataset

- Reads the CSV file into a table. It has 10,000 rows (customers).
- **X** = columns 3 up to (but not including) the last one: *CreditScore, Geography, Gender, Age, Tenure, Balance, NumOfProducts, HasCrCard, IsActiveMember, EstimatedSalary*. That is 10 inputs.
- **y** = the last column, *Exited*: 1 if the customer left, 0 if they stayed. This is what we want to **predict**.

**Why skip the first 3 columns?** They are *RowNumber*, *CustomerId* and *Surname*. They are just labels. A customer's surname doesn't tell you whether they will leave, and keeping them could only confuse the model.

`print(X)` shows that Geography and Gender are still words, like "France" and "Female".

In [0]:
dataset = pd.read_csv('Churn_Modelling.csv')
X = dataset.iloc[:, 3:-1].values
y = dataset.iloc[:, -1].values

In [4]:
print(X)

[[619 'France' 'Female' ... 1 1 101348.88]
 [608 'Spain' 'Female' ... 0 1 112542.58]
 [502 'France' 'Female' ... 1 0 113931.57]
 ...
 [709 'France' 'Female' ... 0 1 42085.58]
 [772 'Germany' 'Male' ... 1 0 92888.52]
 [792 'France' 'Female' ... 1 0 38190.78]]


In [5]:
print(y)

[1 0 1 ... 1 1 0]


### Encoding categorical data

The network can only do maths on numbers, so the two word columns must be turned into numbers. They are handled in two different ways:

**Gender, with label encoding.** There are only two values, so each is simply replaced by a single number: *Female* becomes **0** and *Male* becomes **1** (alphabetical order). With just two options this is safe. It is like a yes/no switch, and there is no false "ordering" to worry about.
- `X[:, 2]` means "column 2 of X" (counting from 0), which is Gender.

**Geography, with one-hot encoding.** There are three countries. Numbering them 0, 1, 2 would wrongly suggest Spain is "twice" Germany. Instead the column is replaced by **three yes/no columns**:

| Geography | France | Germany | Spain |
|---|---|---|---|
| France | 1 | 0 | 0 |
| Germany | 0 | 1 | 0 |
| Spain | 0 | 0 | 1 |

- `[1]` means "apply this to column 1", which is Geography.
- `remainder='passthrough'` means "leave the other columns as they are".
- The new country columns go **at the front** of X. That's why the last `print(X)` starts with `1.0 0.0 0.0` for a French customer. X now has 12 columns.

Label Encoding the "Gender" column

In [0]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
X[:, 2] = le.fit_transform(X[:, 2])

In [7]:
print(X)

[[619 'France' 0 ... 1 1 101348.88]
 [608 'Spain' 0 ... 0 1 112542.58]
 [502 'France' 0 ... 1 0 113931.57]
 ...
 [709 'France' 0 ... 0 1 42085.58]
 [772 'Germany' 1 ... 1 0 92888.52]
 [792 'France' 0 ... 1 0 38190.78]]


One Hot Encoding the "Geography" column

In [0]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
ct = ColumnTransformer(transformers=[('encoder', OneHotEncoder(), [1])], remainder='passthrough')
X = np.array(ct.fit_transform(X))

In [9]:
print(X)

[[1.0 0.0 0.0 ... 1 1 101348.88]
 [0.0 0.0 1.0 ... 0 1 112542.58]
 [1.0 0.0 0.0 ... 1 0 113931.57]
 ...
 [1.0 0.0 0.0 ... 0 1 42085.58]
 [0.0 1.0 0.0 ... 1 0 92888.52]
 [1.0 0.0 0.0 ... 1 0 38190.78]]


### Splitting the dataset into the Training set and Test set

- **80% of the rows (8,000 customers)** go into the *training set*. The network learns from these.
- **20% of the rows (2,000 customers)** go into the *test set*. These are held back to check the network afterwards.
- `random_state = 0` makes the shuffle come out the same every time you run it.

**Why?** Testing on customers the network has never seen shows whether it really learned the pattern rather than memorising the answers.

In [0]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 0)

### Feature Scaling

The columns are on very different scales. Balance can be 100,000 while HasCrCard is only 0 or 1. **Standardisation** rescales every column so its average is 0 and most values fall roughly between -3 and +3.

- `fit_transform(X_train)` *learns* each column's average and spread from the training set, then rescales it.
- `transform(X_test)` rescales the test set using **those same training numbers**, so the test set stays truly "unseen".

**Why?** For neural networks scaling is **essential**, not optional. Without it the big-number columns would drown out the small ones, and training would be slow and unstable.

In [0]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

## Part 2 - Building the ANN

Now we stack the layers of the network, one at a time, like building blocks:

**12 inputs → 6 neurons → 6 neurons → 1 output**

![Network diagram: 12 inputs, two hidden layers of 6 neurons, one sigmoid output](../../../docs/diagrams/neural_network.svg)

*Customer data flows left to right through two hidden layers of 6 neurons to one output: the chance the customer leaves.*

### Initializing the ANN

- `Sequential()` creates an **empty network** whose layers will run one after another in a line. Each layer feeds into the next.
- Nothing is learned yet. It's an empty box ready for layers.

In [0]:
ann = tf.keras.models.Sequential()

### Adding the input layer and the first hidden layer

- `Dense` means a **fully connected** layer: every neuron receives every number from the layer before.
- There is no separate "input layer" line. Keras works out from the data that there are **12 inputs** (the 12 columns of X) the first time it sees it.
- `units=6` puts **6 neurons** in this layer. A **hidden layer** is any layer between the inputs and the final answer. There is no magic rule for the number 6; it is a reasonable starting choice you could experiment with.
- `activation='relu'` (**rectifier**) is the rule each neuron applies to its total: *if it's negative, output 0; otherwise pass it on unchanged*. It's simple, fast, and lets the network learn curved, non-straight-line patterns.

In [0]:
ann.add(tf.keras.layers.Dense(units=6, activation='relu'))

### Adding the second hidden layer

- Exactly the same as the first hidden layer: **6 neurons** with **relu**.
- Its inputs are the 6 outputs of the first hidden layer.

**Why a second layer?** Each extra layer lets the network combine simple patterns into more complex ones. For example, "older *and* inactive *and* living in Germany" might mean a higher risk of leaving. Two or more hidden layers is what makes it "deep" learning.

In [0]:
ann.add(tf.keras.layers.Dense(units=6, activation='relu'))

### Adding the output layer

- `units=1`: there is **one neuron**, because we want **one answer**: will this customer leave?
- `activation='sigmoid'` squashes the total into a number **between 0 and 1**, which we read as a **probability**. For example, 0.8 means "80% chance this customer leaves".

**Why sigmoid here but relu before?** Hidden layers just need to pass useful signals along, and relu is best at that. The final layer must give a probability, and sigmoid always lands between 0 and 1. (With three or more possible answers you would use *softmax* instead.)

In [0]:
ann.add(tf.keras.layers.Dense(units=1, activation='sigmoid'))

## Part 3 - Training the ANN

Training means showing the network the 8,000 training customers over and over, and adjusting its weights each time so its guesses get closer to the real answers.

### Compiling the ANN

`compile` sets **how** the network will learn, before any learning starts:

- `optimizer = 'adam'`: the method for nudging the weights. It's a smart version of **gradient descent**, which means "work out which direction makes the error smaller, and take a small step that way". Adam automatically adjusts the step size as it goes, and it is a strong default choice.
- `loss = 'binary_crossentropy'`: the way of **measuring how wrong** a guess is when there are two possible answers (leave / stay). It punishes confident wrong answers heavily. For example, saying "1% chance of leaving" about someone who did leave costs a lot. (For three or more answers you'd use *categorical_crossentropy*.)
- `metrics = ['accuracy']`: also report the **percentage of correct guesses** while training, so we can watch progress. It doesn't affect the learning itself.

In [0]:
ann.compile(optimizer = 'adam', loss = 'binary_crossentropy', metrics = ['accuracy'])

### Training the ANN on the Training set

- `batch_size = 32`: instead of updating the weights after every single customer, the network looks at **32 customers at a time**, averages its mistakes, then makes one adjustment. 8,000 ÷ 32 = **250 batches**. That's the `250/250` you see on each line.
- `epochs = 100`: one **epoch** is one full pass through all 8,000 training customers. The network goes through the whole training set **100 times**.

**How to read the output:** each line is one epoch.
- **loss** (the error) should go **down**: here from about 0.80 in epoch 1 to about **0.33** by epoch 100.
- **accuracy** should go **up**: from about 52% at the start (no better than guessing) to about **86.6%** at the end.
- After roughly 30–40 epochs the numbers barely change. The network has learned about as much as it can from this data.

In [17]:
ann.fit(X_train, y_train, batch_size = 32, epochs = 100)

Epoch 1/100
250/250 [==============================] - 0s 1ms/step - loss: 0.8037 - accuracy: 0.5185
Epoch 2/100
250/250 [==============================] - 0s 1ms/step - loss: 0.5291 - accuracy: 0.7901
Epoch 3/100
250/250 [==============================] - 0s 1ms/step - loss: 0.4888 - accuracy: 0.7952
Epoch 4/100
250/250 [==============================] - 0s 1ms/step - loss: 0.4668 - accuracy: 0.7979
Epoch 5/100
250/250 [==============================] - 0s 1ms/step - loss: 0.4478 - accuracy: 0.7994
Epoch 6/100
250/250 [==============================] - 0s 1ms/step - loss: 0.4302 - accuracy: 0.8049
Epoch 7/100
250/250 [==============================] - 0s 1ms/step - loss: 0.4123 - accuracy: 0.8119
Epoch 8/100
250/250 [==============================] - 0s 1ms/step - loss: 0.3947 - accuracy: 0.8238
Epoch 9/100
250/250 [==============================] - 0s 1ms/step - loss: 0.3807 - accuracy: 0.8355
Epoch 10/100
250/250 [==============================] - 0s 1ms/step - loss: 0.3720 - accura

## Part 4 - Making the predictions and evaluating the model

Now we use the trained network: first on one made-up customer, then on the 2,000 test customers it has never seen.

### Predicting the result of a single observation

**Homework**

Use our ANN model to predict if the customer with the following informations will leave the bank: 

Geography: France

Credit Score: 600

Gender: Male

Age: 40 years old

Tenure: 3 years

Balance: \$ 60000

Number of Products: 2

Does this customer have a credit card? Yes

Is this customer an Active Member: Yes

Estimated Salary: \$ 50000

So, should we say goodbye to that customer?

**Solution**

The new customer's details must be given **in exactly the same form** as the training data:

- **France** becomes `1, 0, 0` (the three one-hot country columns, at the front).
- Then the other columns in their original order: credit score `600`, Gender **Male** becomes `1`, age `40`, tenure `3`, balance `60000`, products `2`, has a credit card `1` (yes), active member `1` (yes), salary `50000`.
- `sc.transform(...)` applies the **same scaling** learned from the training set. Without this, the network would get numbers in a completely different range from what it learned on.
- `ann.predict(...)` returns a probability between 0 and 1 (from the sigmoid).
- `> 0.5` turns it into a yes/no answer: **above 50% means "will leave" (True)**, otherwise "will stay" (False).

The output is `[[False]]`: the network predicts this customer **stays**.

In [18]:
print(ann.predict(sc.transform([[1, 0, 0, 600, 1, 40, 3, 60000, 2, 1, 1, 50000]])) > 0.5)

[[False]]


Therefore, our ANN model predicts that this customer stays in the bank!

**Important note 1:** Notice that the values of the features were all input in a double pair of square brackets. That's because the "predict" method always expects a 2D array as the format of its inputs. And putting our values into a double pair of square brackets makes the input exactly a 2D array.

**Important note 2:** Notice also that the "France" country was not input as a string in the last column but as "1, 0, 0" in the first three columns. That's because of course the predict method expects the one-hot-encoded values of the state, and as we see in the first row of the matrix of features X, "France" was encoded as "1, 0, 0". And be careful to include these values in the first three columns, because the dummy variables are always created in the first columns.

### Predicting the Test set results

- `ann.predict(X_test)` gives a leaving probability for each of the 2,000 test customers.
- `y_pred > 0.5` turns each probability into **True/False** (shown as 1/0). The rule is the same as before: over 50% means "will leave".
- The printout is a two-column table: **predicted | real**. A row like `[0 1]` is a miss: the network said "stays" but the customer actually left.

In [19]:
y_pred = ann.predict(X_test)
y_pred = (y_pred > 0.5)
print(np.concatenate((y_pred.reshape(len(y_pred),1), y_test.reshape(len(y_test),1)),1))

[[0 0]
 [0 1]
 [0 0]
 ...
 [0 0]
 [0 0]
 [0 0]]


### Making the Confusion Matrix

A **confusion matrix** counts the right and wrong guesses. Rows are the **real** answer and columns are the **predicted** answer:

| | Predicted: stays | Predicted: leaves |
|---|---|---|
| **Really stayed** | **1516** ✓ | 79 ✗ |
| **Really left** | 200 ✗ | **205** ✓ |

- **1,516 + 205 = 1,721** correct out of 2,000, so the **accuracy is 0.8605 (86%)**. That matches the training accuracy, so the network didn't just memorise the training data.
- The weak spot is the **200 customers who left but were predicted to stay**. The network catches only about half of the leavers (205 out of 405). That is partly because only about 1 in 5 customers leaves, so "stays" is the safer guess.

**In one sentence:** turn the bank's customer details into scaled numbers, build a small network (12 inputs → 6 → 6 → 1), train it for 100 passes to predict who will leave, and check it on 2,000 unseen customers, where it is right about 86% of the time.

In [20]:
from sklearn.metrics import confusion_matrix, accuracy_score
cm = confusion_matrix(y_test, y_pred)
print(cm)
accuracy_score(y_test, y_pred)

[[1516   79]
 [ 200  205]]


0.8605

## Common mistakes to avoid
- **Skipping feature scaling.** Without `StandardScaler`, big columns like Balance (up to six figures) swamp small ones like HasCrCard (0 or 1), and training becomes slow and unstable. For neural networks, always scale the inputs.
- **Feeding a new customer in the wrong format.** `predict` expects the exact column layout of the training data: the three one-hot country columns first, Gender as 0/1, then the rest in order, all wrapped in `[[...]]` and passed through `sc.transform`. Typing `'France'` or forgetting `sc.transform` gives an error or a meaningless answer.
- **Calling `sc.fit_transform` on the test set or on a single new customer.** That re-learns the averages from different data, so the scaling no longer matches what the network learned on. Use `fit_transform` once on `X_train`, and only `transform` afterwards.
- **Trusting accuracy alone.** About 4 in 5 customers stay, so a model that always says "stays" already scores around 80%. Check the confusion matrix too: here 200 of the 405 real leavers were missed.

## Try this yourself
Change one thing, rerun the notebook, and see what happens.

1. In the training cell, change `epochs = 100` to `epochs = 10`. First re-run the build cells from `ann = tf.keras.models.Sequential()` onward so you start from a fresh network, then retrain and re-run everything below. What to notice: the final training accuracy and the test accuracy from `accuracy_score`. Is the score much worse, or did most of the learning happen early?
2. In both hidden-layer cells, change `units=6` to `units=32`, rebuild the network from `Sequential()` and retrain. What to notice: whether the training accuracy creeps above the test accuracy. A growing gap between them is a sign the network is starting to memorise the training customers.
3. In the "Predicting the Test set results" cell, change `y_pred = (y_pred > 0.5)` to `y_pred = (y_pred > 0.3)` and re-run it and the confusion matrix cell. What to notice: the bottom-right number (leavers correctly caught) should go up, but the top-right number (stayers wrongly flagged) also goes up, and accuracy may drop. A lower cut-off trades more false alarms for catching more leavers.
4. In the single-prediction cell, remove `> 0.5` so it prints the raw probability, and change the age from `40` to `60` (the 6th number): `print(ann.predict(sc.transform([[1, 0, 0, 600, 1, 60, 3, 60000, 2, 1, 1, 50000]])))`. What to notice: how the predicted chance of leaving changes when only the age changes.

## Check your understanding
Try to answer in your head first, then click to reveal.

**1. Why does the output layer use `sigmoid` while the hidden layers use `relu`?**

<details><summary>Show answer</summary>

The output must be a probability, and sigmoid always gives a number between 0 and 1. The hidden layers just need to pass useful signals along, and relu (negative becomes 0, positive passes through) is simple and fast for that.

</details>

**2. Each training line shows `250/250`. Where does 250 come from?**

<details><summary>Show answer</summary>

There are 8,000 training customers and `batch_size = 32`, so one epoch is 8,000 ÷ 32 = 250 batches. The network updates its weights once per batch, so 250 times per epoch.

</details>

**3. The confusion matrix is `[[1516 79] [200 205]]`. What does the 200 mean, and why does it matter for the bank?**

<details><summary>Show answer</summary>

200 customers really left but the network predicted they would stay. Those are exactly the customers the bank wants to catch, so even with 86% accuracy the model misses about half of the leavers (it catches 205 of 405).

</details>

**4. What would happen if you removed the second hidden layer and kept everything else the same?**

<details><summary>Show answer</summary>

The network would still train, and on a fairly simple table like this it may score in a similar range, because one hidden layer can already learn a lot. It would just have less ability to combine simple patterns into more complex ones. Try it and compare the accuracy.

</details>

---
📚 [Learning path](../../../LEARNING_PATH.md) · [Glossary](../../../GLOSSARY.md)